# Triển khai "Chất lượng khách hàng"

Đầu tiên đọc cái file data cần thiết và nối các bảng lại với nhau để có cái nhìn xuyên suốt

In [ ]:
import pandas as pd
import glob

df_customers = pd.read_csv('../../data/customers.csv')
df_orders = pd.read_csv('../../data/orders.csv')

# Kết hợp (Join) để biết mỗi khách hàng đến từ kênh nào
df_master = pd.merge(df_orders, df_customers, on='customer_id', how='left')
df_master['order_date'] = pd.to_datetime(df_master['order_date'])


Tiếp theo là tính toán các chỉ số "chất lượng" 

- Frequency (F): Tổng số đơn hàng trên mỗi khách hàng.

- Retention: Khách hàng có quay lại sau đơn hàng đầu tiên không?

- CLV (Customer Lifetime Value): Tổng giá trị mua sắm (nếu có cột giá) hoặc dùng số đơn hàng tích lũy làm proxy.

- Inter-order Gap: Khoảng cách trung bình giữa các lần mua (để chứng minh khách Organic mua hàng đều đặn hơn).

In [6]:
# Tính số đơn hàng và thời gian gắn bó của mỗi khách hàng
customer_metrics = df_master.groupby(['customer_id', 'acquisition_channel']).agg(
    total_orders=('order_id', 'count'),
    first_purchase=('order_date', 'min'),
    last_purchase=('order_date', 'max')
).reset_index()

# Tính số ngày gắn bó (Tenure)
customer_metrics['tenure_days'] = (customer_metrics['last_purchase'] - customer_metrics['first_purchase']).dt.days

Tính toán trung bình các chỉ số trên theo từng acquisition_channel.

In [ ]:
# Phân tích theo kênh
channel_performance = customer_metrics.groupby('acquisition_channel').agg(
    avg_orders=('total_orders', 'mean'),
    repeat_rate=('total_orders', lambda x: (x > 1).mean() * 100),
    avg_tenure=('tenure_days', 'mean')
).sort_values('avg_orders', ascending=False)

print(channel_performance)

                     avg_orders  repeat_rate   avg_tenure
acquisition_channel                                      
organic_search         7.206939    75.384045  1764.946679
social_media           7.193534    75.674925  1777.578936
paid_search            7.164454    75.020834  1759.397633
email_campaign         7.144498    74.765754  1748.483741
referral               7.106812    75.132275  1752.889330
direct                 7.089955    74.839853  1747.331471


In [ ]:
# Tính toán các chỉ số phái sinh
customer_metrics['tenure_days'] = (customer_metrics['last_purchase'] - customer_metrics['first_purchase']).dt.days
customer_metrics['is_repeat_customer'] = customer_metrics['total_orders'].apply(lambda x: 1 if x > 1 else 0)

In [10]:
df_master.to_csv('df_master.csv', index=False, encoding='utf-8-sig')